In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub


In [2]:
!ls /kaggle/input/competitions/multi-script-emotion-classification-t-2-2026


competition_test.csv   competition_val.csv
competition_train.csv  sample_submission.csv


## Setup

In [3]:
DATA_DIR = "/kaggle/input/competitions/multi-script-emotion-classification-t-2-2026" 
train_df = pd.read_csv(os.path.join(DATA_DIR, "competition_train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR,"competition_test.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "competition_val.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

In [4]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_sub.shape)

Train shape: (7176, 4)
Val shape: (2392, 4)
Test shape: (2392, 3)
Sample submission shape: (2392, 2)


In [5]:
print("\nTrain columns:", train_df.columns.tolist())


Train columns: ['id', 'Sentence', 'language', 'emotion']


In [6]:
train_df.head(10)

,id,Sentence,language,emotion
0,1801,رُک، بہٕ چھس صرف یہ یقینی بناونٕچ کوٗشش کران ز...,Kashmiri,disgust
1,7895,بہٕ چھس مایوس گژھان ییٚلہ بہٕ بٹوار دۄہ شامس ا...,Kashmiri,anger
2,3369,ꯑꯩꯒꯤ ꯑꯉꯥꯡ ꯌꯣꯛꯄꯗ ꯆꯥꯗꯤꯡꯁꯤ ꯀꯌꯥ ꯌꯥꯝꯅ ꯍꯦꯟꯒꯠꯂꯛꯂꯤꯕꯒꯦ ...,Manipuri,sad
3,5191,بہٕ چھس بیٚنٛکس مٗقابلہٕ جٲتی سودٕکہِ شرح پیش...,Kashmiri,anger
4,7934,کوچ سٕنٛز سپورٹس مینشِپٕچ کٔمی چھےٚ واریاہ مای...,Kashmiri,disgust
5,4704,ꯃꯐꯝ ꯑꯗꯨꯒꯤ ꯄꯥꯔꯛꯁꯤꯡ ꯑꯗꯨꯅ ꯁꯍꯔꯒꯤ ꯏꯔꯥꯡ ꯂꯥꯡꯕ ꯄꯨꯟꯁꯤꯗꯒ...,Manipuri,happy
6,5381,"ᱟᱨᱦᱚᱸ ᱱᱟᱯᱟᱭ ᱥᱮ ᱵᱟᱝ, ᱤᱧ ᱢᱤᱫᱴᱟᱝ ᱥᱮᱨᱢᱟᱠᱤᱭᱟᱹ ᱤᱱᱴᱟᱨ...",Santali,fear
7,8221,مےٚ گو خوف ییٚلہ بہٕ اکہ خطرناک علاقہٕ مٔنٛزۍ...,Kashmiri,fear
8,3877,ᱩᱱᱤᱭᱟᱜ ᱱᱚᱣᱟ ᱵᱮᱭᱟᱢ ᱠᱚᱨᱟᱣ ᱦᱚᱨᱟ ᱫᱚ ᱱᱩᱱᱟᱹᱜ ᱜᱮ ᱦᱟᱦᱟ...,Santali,disgust
9,9728,ᱛᱷᱟᱹᱱᱤᱭᱚ ᱠᱟᱹᱨᱜᱚᱞ ᱠᱚᱣᱟᱜ ᱱᱤᱯᱷᱩᱴ ᱠᱟᱹᱢᱤ ᱟᱨ ᱦᱩᱱᱟᱹᱨ ...,Santali,surprise


In [7]:
# Check for missing values
print("Missing values in train:\n", train_df.isnull().sum())
print("\nMissing values in val:\n", val_df.isnull().sum())
print("\nMissing values in test:\n", test_df.isnull().sum())

Missing values in train:
 id          0
Sentence    0
language    0
emotion     0
dtype: int64

Missing values in val:
 id          0
Sentence    0
language    0
emotion     0
dtype: int64

Missing values in test:
 id          0
Sentence    0
language    0
dtype: int64


In [8]:
print("\nEmotion distribution (train):")
print(train_df['emotion'].value_counts())


Emotion distribution (train):
emotion
fear        1640
happy       1225
surprise    1132
sad         1122
anger       1097
disgust      960
Name: count, dtype: int64


In [9]:
print("\nEmotion distribution (val):")
print(val_df['emotion'].value_counts())


Emotion distribution (val):
emotion
fear        546
happy       407
surprise    377
sad         375
anger       366
disgust     321
Name: count, dtype: int64


In [10]:
print("Language distribution (train):")
print(train_df['language'].value_counts())

Language distribution (train):
language
Santali     2551
Kashmiri    2367
Manipuri    2258
Name: count, dtype: int64


In [11]:
print("\nCross-tab of language vs emotion:")
print(pd.crosstab(train_df['language'], train_df['emotion']))


Cross-tab of language vs emotion:
emotion   anger  disgust  fear  happy  sad  surprise
language                                            
Kashmiri    352      294   626    389  350       356
Manipuri    371      332   367    416  385       387
Santali     374      334   647    420  387       389


In [12]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets scikit-learn sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 101.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 111.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

In [14]:
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [15]:
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB): {torch.cuda.get_device_properties(0).total_memory/1e9}")

GPU: Tesla T4
VRAM (GB): 15.636037632


In [16]:
MODEL_NAME = "google/gemma-3-1b-it"

In [17]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient


In [18]:
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [20]:
for lang in train_df['language'].unique():
    sample_text = train_df[train_df['language'] == lang]['Sentence'].iloc[0]
    tokens = tokenizer.tokenize(sample_text)
    print(f"Language: {lang}")
    print(f"Text: {sample_text}")
    print(f"Number of Tokens: {len(tokens)}")
    print(f"Tokens: {tokens[:20]}")

Language: Kashmiri
Text: رُک، بہٕ چھس صرف یہ یقینی بناونٕچ کوٗشش کران ز مےٚ چھ فراہم کرنہٕ آمتیٚن خدماتن خٲطرٕ معقول قۭمت میلان۔
Number of Tokens: 51
Tokens: ['ر', 'ُ', 'ک', '،', '▁بہ', 'ٕ', '▁چھ', 'س', '▁صرف', '▁یہ', '▁یقینی', '▁بنا', 'ون', 'ٕ', 'چ', '▁کو', 'ٗ', 'شش', '▁کر', 'ان']
Language: Manipuri
Text: ꯑꯩꯒꯤ ꯑꯉꯥꯡ ꯌꯣꯛꯄꯗ ꯆꯥꯗꯤꯡꯁꯤ ꯀꯌꯥ ꯌꯥꯝꯅ ꯍꯦꯟꯒꯠꯂꯛꯂꯤꯕꯒꯦ ꯍꯥꯏꯕꯗꯨ ꯎꯕꯗ ꯇꯁꯦꯡꯅ ꯄꯨꯛꯅꯤꯡ ꯁꯣꯟꯊꯩ꯫
Number of Tokens: 150
Tokens: ['ꯑ', '<0xEA>', '<0xAF>', '<0xA9>', '<0xEA>', '<0xAF>', '<0x92>', 'ꯤ', '▁', 'ꯑ', '<0xEA>', '<0xAF>', '<0x89>', '<0xEA>', '<0xAF>', '<0xA5>', 'ꯡ', '▁', 'ꯌ', 'ꯣ']
Language: Santali
Text: ᱟᱨᱦᱚᱸ ᱱᱟᱯᱟᱭ ᱥᱮ ᱵᱟᱝ, ᱤᱧ ᱢᱤᱫᱴᱟᱝ ᱥᱮᱨᱢᱟᱠᱤᱭᱟᱹ ᱤᱱᱴᱟᱨᱱᱮᱴ ᱥᱟᱵᱥᱠᱨᱤᱯᱥᱚᱱ ᱞᱟᱹᱜᱤᱫ ᱥᱮᱱ ᱟᱠᱟᱱᱟ, ᱱᱚᱣᱟ ᱩᱭᱦᱟᱹᱨ ᱵᱤᱨᱩᱫᱷᱨᱮ ᱡᱮ, ᱱᱚᱶᱟ ᱫᱚ ᱤᱧᱟᱹᱜ ᱚᱯᱷᱤᱥ ᱠᱚᱞ ᱫᱚ ᱵᱟᱝ ᱦᱟᱹᱱᱼᱟ ᱾
Number of Tokens: 144
Tokens: ['ᱟ', 'ᱨ', 'ᱦ', 'ᱚ', 'ᱸ', '▁', 'ᱱ', 'ᱟ', 'ᱯ', 'ᱟ', 'ᱭ', '▁', 'ᱥ', 'ᱮ', '▁', 'ᱵ', 'ᱟ', 'ᱝ', ',', '▁']


In [21]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [22]:
print(model.config)

Gemma3TextConfig {
  "_sliding_window_pattern": 6,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": [
    1,
    106
  ],
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 1152,
  "initializer_range": 0.02,
  "intermediate_size": 6912,
  "layer_types": [
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
 

In [23]:
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())/1e9}B")

Number of parameters: 0.999885952B


In [24]:
import re
from sklearn.preprocessing import LabelEncoder

In [25]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    # Collapse multiple whitespaces/newlines
    text = re.sub(r'\s+', ' ', text)
    # Remove obvious URL/HTML noise if present (rare here but safe to include)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    return text.strip()

In [26]:
train_df['clean_sentence'] = train_df['Sentence'].apply(clean_text)
val_df['clean_sentence']   = val_df['Sentence'].apply(clean_text)
test_df['clean_sentence']  = test_df['Sentence'].apply(clean_text)

In [27]:
print("Empty train sentences after cleaning:", (train_df['clean_sentence'] == "").sum())
print("Empty val sentences after cleaning:", (val_df['clean_sentence'] == "").sum())
print("Empty test sentences after cleaning:", (test_df['clean_sentence'] == "").sum())

Empty train sentences after cleaning: 0
Empty val sentences after cleaning: 0
Empty test sentences after cleaning: 0


In [28]:
EMOTION_CLASSES = ['fear', 'happy', 'surprise', 'sad', 'anger', 'disgust']
label2id = {label: idx for idx, label in enumerate(EMOTION_CLASSES)}
id2label = {idx: label for label, idx in label2id.items()}

In [29]:
train_df['label'] = train_df['emotion'].map(label2id)
val_df['label']   = val_df['emotion'].map(label2id)

In [30]:
print("Unmapped labels in train:", train_df['label'].isna().sum())
print("Unmapped labels in val:", val_df['label'].isna().sum())
print("\nLabel distribution (train):")
print(train_df['label'].value_counts().sort_index())

Unmapped labels in train: 0
Unmapped labels in val: 0

Label distribution (train):
label
0    1640
1    1225
2    1132
3    1122
4    1097
5     960
Name: count, dtype: int64


In [31]:
LANGUAGE_NAMES = {
    'santali': 'Santali',
    'kashmiri': 'Kashmiri',
    'manipuri': 'Manipuri',
}

In [32]:
def build_prompt(sentence, language):
    lang_name = LANGUAGE_NAMES.get(str(language).lower(), str(language))
    prompt = (
        f"You are an expert emotion classifier for {lang_name} text.\n"
        f"Classify the emotion expressed in the following {lang_name} sentence "
        f"into exactly one of: fear, happy, surprise, sad, anger, disgust.\n\n"
        f"Sentence: {sentence}\n"
        f"Emotion:"
    )
    return prompt

In [33]:
# Check actual unique language values in the data
print("Unique language values:", train_df['language'].unique())

train_df['prompt'] = train_df.apply(lambda r: build_prompt(r['clean_sentence'], r['language']), axis=1)
val_df['prompt']   = val_df.apply(lambda r: build_prompt(r['clean_sentence'], r['language']), axis=1)
test_df['prompt']  = test_df.apply(lambda r: build_prompt(r['clean_sentence'], r['language']), axis=1)

print("\nExample prompt:\n")
print(train_df['prompt'].iloc[0])

Unique language values: ['Kashmiri' 'Manipuri' 'Santali']

Example prompt:

You are an expert emotion classifier for Kashmiri text.
Classify the emotion expressed in the following Kashmiri sentence into exactly one of: fear, happy, surprise, sad, anger, disgust.

Sentence: رُک، بہٕ چھس صرف یہ یقینی بناونٕچ کوٗشش کران ز مےٚ چھ فراہم کرنہٕ آمتیٚن خدماتن خٲطرٕ معقول قۭمت میلان۔
Emotion:


In [34]:
# Check sequence length distribution to decide max_length for tokenization
train_df['token_len'] = train_df['prompt'].apply(lambda x: len(tokenizer.tokenize(x)))
print(train_df['token_len'].describe())
print("\n95th percentile:", train_df['token_len'].quantile(0.95))
print("99th percentile:", train_df['token_len'].quantile(0.99))

count    7176.000000
mean      162.662207
std        69.417313
min        54.000000
25%       103.000000
50%       152.000000
75%       204.000000
max       534.000000
Name: token_len, dtype: float64

95th percentile: 295.0
99th percentile: 357.0


In [35]:
from torch.utils.data import Dataset, DataLoader

In [36]:
MAX_LENGTH = 534 

# Make sure tokenizer has a pad token (Gemma's tokenizer usually does, but confirm)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Pad token:", tokenizer.pad_token, "| Pad token id:", tokenizer.pad_token_id)

Pad token: <pad> | Pad token id: 0


In [37]:
class EmotionDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=MAX_LENGTH):
        self.prompts = list(prompts)
        self.labels = list(labels) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.prompts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [38]:
train_dataset = EmotionDataset(
    train_df['prompt'].tolist(),
    train_df['label'].astype(int).tolist(),
    tokenizer
)

In [39]:
val_dataset = EmotionDataset(
    val_df['prompt'].tolist(),
    val_df['label'].astype(int).tolist(),
    tokenizer
)

In [40]:
test_dataset = EmotionDataset(
    test_df['prompt'].tolist(),
    None,
    tokenizer
)

In [41]:
print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 7176
Val dataset size: 2392
Test dataset size: 2392


In [42]:
sample = train_dataset[0]
print("\nSample keys:", sample.keys())
print("input_ids shape:", sample['input_ids'].shape)
print("attention_mask shape:", sample['attention_mask'].shape)
print("labels:", sample['labels'])


Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([534])
attention_mask shape: torch.Size([534])
labels: tensor(5)


In [43]:
BATCH_SIZE = 4  
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [44]:
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1794
Val batches: 598
Test batches: 598


In [45]:
batch = next(iter(train_loader))
print("\nBatch input_ids shape:", batch['input_ids'].shape)
print("Batch labels shape:", batch['labels'].shape)


Batch input_ids shape: torch.Size([4, 534])
Batch labels shape: torch.Size([4])


In [46]:
from peft import LoraConfig, get_peft_model, TaskType

In [47]:
NUM_LABELS = len(EMOTION_CLASSES)

In [48]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer.pad_token_id

print(base_model.config)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[transformers] Gemma3TextForSequenceClassification LOAD REPORT from: google/gemma-3-1b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Gemma3TextConfig {
  "_sliding_window_pattern": 6,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": [
    1,
    106
  ],
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 1152,
  "id2label": {
    "0": "fear",
    "1": "happy",
    "2": "surprise",
    "3": "sad",
    "4": "anger",
    "5": "disgust"
  },
  "initializer_range": 0.02,
  "intermediate_size": 6912,
  "label2id": {
    "anger": 4,
    "disgust": 5,
    "fear": 0,
    "happy": 1,
    "sad": 3,
    "surprise": 2
  },
  "layer_types": [
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention

In [49]:
for name, module in base_model.named_modules():
    if any(x in name for x in ["proj"]):
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

In [50]:
# Upgrade torchao to satisfy peft's version check (or uninstall since we don't need quantization)
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.4 MB/s eta 0:00:00:00:01


In [51]:
# LoRA config — target attention + MLP projections for strong adaptation with few trainable params
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                      # rank — 16 is a good starting point for 1B models
    lora_alpha=32,             # scaling factor, typically 2x rank
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 13,052,672 || all params: 1,012,945,536 || trainable%: 1.2886


In [52]:
# Confirm trainable vs frozen parameter split
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.4f}% of total)")

# Move to device explicitly if needed (usually handled by device_map="auto")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Trainable params: 13,052,672 (1.2886% of total)
Using device: cuda


In [53]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (if any):", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected — check Kaggle Settings > Accelerator (should be GPU T4 x2 or P100)")

# Also confirm where the model currently lives
print("\nModel device (sample param):", next(model.parameters()).device)
print("Model dtype (sample param):", next(model.parameters()).dtype)

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version (if any): 12.8
GPU: Tesla T4

Model device (sample param): cuda:0
Model dtype (sample param): torch.bfloat16


In [54]:
from sklearn.metrics import f1_score, classification_report
import numpy as np
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

EPOCHS = 5
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.06

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Total training steps:", total_steps)
print("Warmup steps:", warmup_steps)

Total training steps: 8970
Warmup steps: 538


In [55]:
# Optional: class-weighted loss to counter imbalance (check if needed based on Step 1's distribution)
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_LABELS),
    y=train_df['label'].astype(int).values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.bfloat16).to(device)
print("Class weights:", dict(zip(EMOTION_CLASSES, class_weights.round(3))))

Class weights: {'fear': np.float64(0.729), 'happy': np.float64(0.976), 'surprise': np.float64(1.057), 'sad': np.float64(1.066), 'anger': np.float64(1.09), 'disgust': np.float64(1.246)}


In [56]:
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = total_loss / len(dataloader)
    return avg_loss, macro_f1, all_preds, all_labels

In [57]:
best_macro_f1 = 0.0
best_model_path = "/kaggle/working/best_lora_gemma"

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Weighted cross-entropy loss for class imbalance
        loss = F.cross_entropy(logits.float(), labels, weight=class_weights_tensor.float())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if step % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_macro_f1, val_preds, val_labels = evaluate(model, val_loader)

    print(f"\n=== Epoch {epoch+1} Summary ===")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}\n")

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"✅ New best model saved (Macro F1: {best_macro_f1:.4f})")

print(f"\nTraining complete. Best Val Macro F1: {best_macro_f1:.4f}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch 1 | Step 0/1794 | Loss: 2.1641
Epoch 1 | Step 50/1794 | Loss: 2.0130
Epoch 1 | Step 100/1794 | Loss: 2.0455
Epoch 1 | Step 150/1794 | Loss: 1.8417
Epoch 1 | Step 200/1794 | Loss: 2.3756
Epoch 1 | Step 250/1794 | Loss: 1.9404
Epoch 1 | Step 300/1794 | Loss: 2.0542
Epoch 1 | Step 350/1794 | Loss: 1.6572
Epoch 1 | Step 400/1794 | Loss: 2.1916
Epoch 1 | Step 450/1794 | Loss: 1.7295
Epoch 1 | Step 500/1794 | Loss: 1.6960
Epoch 1 | Step 550/1794 | Loss: 2.2935
Epoch 1 | Step 600/1794 | Loss: 1.5244
Epoch 1 | Step 650/1794 | Loss: 1.9598
Epoch 1 | Step 700/1794 | Loss: 1.9192
Epoch 1 | Step 750/1794 | Loss: 2.3698
Epoch 1 | Step 800/1794 | Loss: 2.0332
Epoch 1 | Step 850/1794 | Loss: 1.9992
Epoch 1 | Step 900/1794 | Loss: 1.5710
Epoch 1 | Step 950/1794 | Loss: 1.9476
Epoch 1 | Step 1000/1794 | Loss: 1.7788
Epoch 1 | Step 1050/1794 | Loss: 1.8543
Epoch 1 | Step 1100/1794 | Loss: 1.7392
Epoch 1 | Step 1150/1794 | Loss: 2.4192
Epoch 1 | Step 1200/1794 | Loss: 1.7745
Epoch 1 | Step 1250/179

In [58]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [59]:
best_model_path = "/kaggle/working/best_lora_gemma"

# Reload base model fresh, then attach the saved LoRA adapter
eval_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
eval_base_model.config.pad_token_id = tokenizer.pad_token_id

eval_model = PeftModel.from_pretrained(eval_base_model, best_model_path)
eval_model.eval()

print("Best model loaded successfully.")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[transformers] Gemma3TextForSequenceClassification LOAD REPORT from: google/gemma-3-1b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Best model loaded successfully.


In [60]:
val_loss, val_macro_f1, val_preds, val_labels = evaluate(eval_model, val_loader)

print(f"Final Val Macro F1: {val_macro_f1:.4f}")
print(f"Final Val Loss: {val_loss:.4f}")

print("\nClassification Report:")
print(classification_report(
    val_labels, val_preds,
    target_names=EMOTION_CLASSES,
    digits=4
))

Final Val Macro F1: 0.5332
Final Val Loss: 1.4877

Classification Report:
              precision    recall  f1-score   support

        fear     0.6493    0.6172    0.6329       546
       happy     0.5793    0.6192    0.5986       407
    surprise     0.6431    0.6021    0.6219       377
         sad     0.4506    0.5227    0.4840       375
       anger     0.4694    0.4617    0.4656       366
     disgust     0.4172    0.3769    0.3961       321

    accuracy                         0.5443      2392
   macro avg     0.5348    0.5333    0.5332      2392
weighted avg     0.5466    0.5443    0.5446      2392



In [61]:
# Confusion matrix — see which emotions get confused with each other
cm = confusion_matrix(val_labels, val_preds)
print("Confusion Matrix (rows=true, cols=predicted):")
print("Classes:", EMOTION_CLASSES)
print(cm)

Confusion Matrix (rows=true, cols=predicted):
Classes: ['fear', 'happy', 'surprise', 'sad', 'anger', 'disgust']
[[337  40  26  55  46  42]
 [ 26 252  27  55  23  24]
 [ 35  37 227  22  34  22]
 [ 50  52  17 196  27  33]
 [ 37  30  29  53 169  48]
 [ 34  24  27  54  61 121]]


In [62]:
# Per-language breakdown — critical since this is a cross-lingual task
val_df_eval = val_df.copy()
val_df_eval['pred'] = [id2label[p] for p in val_preds]
val_df_eval['true'] = [id2label[l] for l in val_labels]

for lang in val_df_eval['language'].unique():
    subset = val_df_eval[val_df_eval['language'] == lang]
    lang_f1 = f1_score(subset['true'], subset['pred'], average='macro', labels=EMOTION_CLASSES)
    print(f"{lang}: Macro F1 = {lang_f1:.4f} (n={len(subset)})")

Santali: Macro F1 = 0.4837 (n=851)
Kashmiri: Macro F1 = 0.5437 (n=788)
Manipuri: Macro F1 = 0.5666 (n=753)


In [63]:
# Inspect a few misclassified examples per language for qualitative insight
misclassified = val_df_eval[val_df_eval['pred'] != val_df_eval['true']]
print(f"Total misclassified: {len(misclassified)} / {len(val_df_eval)}")

for lang in misclassified['language'].unique():
    print(f"\n--- {lang} misclassifications (sample) ---")
    sample = misclassified[misclassified['language'] == lang].head(3)
    for _, row in sample.iterrows():
        print(f"True: {row['true']} | Pred: {row['pred']} | Text: {row['Sentence'][:100]}")

Total misclassified: 1090 / 2392

--- Santali misclassifications (sample) ---
True: disgust | Pred: anger | Text: ᱞᱚᱠᱷᱱᱳ ᱨᱮᱭᱟᱜ ᱯᱟᱵᱞᱤᱠ ᱥᱟᱜᱟᱲᱚᱢ ᱵᱚᱱᱫᱮᱡ ᱨᱮ ᱥᱟᱯᱷᱟᱼᱥᱟᱹᱯᱷᱤ ᱨᱮᱭᱟᱜ ᱟᱱᱟᱴ ᱫᱚ ᱟᱹᱲᱤᱥ ᱜᱮᱭᱟ ᱾
True: surprise | Pred: anger | Text: ᱟᱢ ᱫᱚ ᱪᱮᱫ ᱵᱟᱢ ᱵᱟᱰᱟᱭᱟ ᱡᱮ ᱤᱧ ᱓ ᱪᱟᱸᱫᱚ ᱞᱟᱦᱟᱨᱮ ᱠᱟᱹᱣᱰᱤ ᱮᱢ ᱨᱮᱭᱟᱜ ᱠᱟᱛᱷᱟ ᱮᱢ ᱠᱟᱛᱮᱫ ᱚᱱᱞᱟᱭᱤᱱ ᱞᱤᱝᱠ ᱨᱮ ᱠᱞᱤᱠᱼᱤᱧ ᱵᱚᱱ
True: disgust | Pred: surprise | Text: ᱤᱧ ᱫᱚ ᱵᱷᱟᱲᱩᱢᱵᱷᱟᱜ ᱞᱤᱱᱟᱹᱧ ! ᱚᱱᱟ ᱫᱟᱱ ᱫᱚ ᱱᱚᱝᱠᱟᱱ ᱡᱟᱦᱟᱱᱟᱜ ᱫᱚ ᱵᱟᱝ ᱠᱟᱱᱟ ᱡᱟᱦᱟ ᱫᱚ ᱟᱵᱚ ᱚᱱᱟ ᱛᱟᱭᱚᱢ ᱛᱮᱵᱚᱱ ᱧᱟᱢ ᱫᱟᱲᱮ

--- Kashmiri misclassifications (sample) ---
True: fear | Pred: anger | Text: مےٚ اوس پننس شر سند اکاونٹ قٲیم کرنس منٛز کینٛہہ کھوژُن تہٕ مےٚ ووٚن بینک منیجرس ز بہٕ چھس نہٕ یژھان
True: surprise | Pred: anger | Text: تہند ردِ عمل نہ ہاونس پیٹھ  گوس بہ ہراسان تہ سنجیدہ پٲٹھی سوچُم زِ توٚہی طالب علم کرِو نویٛن خیالن ا
True: anger | Pred: sad | Text: بہٕ ووتس اکھ گٲنٛٹہٕ برونٛہہ جویلری سٹورس منٛز تہٕ ونیک تام آو نہ کانہہ مےٚ نش۔ مےٚ تروو تُہند سٹور 

--- Manipuri misclassifications (sample) ---
True: sad | Pred: anger | Text: ꯑ

In [64]:
import torch
from tqdm.auto import tqdm

eval_model.eval()
all_test_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Running inference on test set"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = eval_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_test_preds.extend(preds.cpu().numpy())

print(f"Total test predictions: {len(all_test_preds)}")
print(f"Expected test set size: {len(test_df)}")
assert len(all_test_preds) == len(test_df), "Mismatch between predictions and test set size!"

Running inference on test set:   0%|          | 0/598 [00:00<?, ?it/s]

Total test predictions: 2392
Expected test set size: 2392


In [65]:
# Map predicted ids back to emotion label strings
test_df['emotion'] = [id2label[p] for p in all_test_preds]

# Sanity check: prediction distribution (compare to train distribution for a rough sniff test)
print("Predicted emotion distribution on test set:")
print(test_df['emotion'].value_counts())
print("\nTrain emotion distribution (for comparison):")
print(train_df['emotion'].value_counts())

Predicted emotion distribution on test set:
emotion
fear        521
happy       436
sad         434
anger       354
surprise    343
disgust     304
Name: count, dtype: int64

Train emotion distribution (for comparison):
emotion
fear        1640
happy       1225
surprise    1132
sad         1122
anger       1097
disgust      960
Name: count, dtype: int64


In [66]:
# Check for any per-language skew in predictions (useful sanity check)
print("\nPredicted distribution by language:")
print(pd.crosstab(test_df['language'], test_df['emotion']))


Predicted distribution by language:
emotion   anger  disgust  fear  happy  sad  surprise
language                                            
Kashmiri    106       83   222    124  144       111
Manipuri    128      122    80    152  146       124
Santali     120       99   219    160  144       108


In [67]:
# Build submission file matching required format: id, emotion
submission = test_df[['id', 'emotion']].copy()

# Verify against sample_submission format
print("Sample submission columns:", sample_sub.columns.tolist())
print("Sample submission head:\n", sample_sub.head())
print("\nOur submission head:\n", submission.head())

# Verify all ids match expected set and no missing/duplicate ids
print("\nSubmission shape:", submission.shape)
print("Unique ids:", submission['id'].nunique())
print("Any missing emotion values:", submission['emotion'].isna().sum())
print("All emotions valid:", submission['emotion'].isin(EMOTION_CLASSES).all())

Sample submission columns: ['id', 'emotion']
Sample submission head:
      id   emotion
0   556       sad
1  1213     anger
2   744  surprise
3   443     anger
4  7673     anger

Our submission head:
      id emotion
0   556    fear
1  1213   happy
2   744   happy
3   443   anger
4  7673   happy

Submission shape: (2392, 2)
Unique ids: 2392
Any missing emotion values: 0
All emotions valid: True


In [68]:
# Save final submission
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")

# Final verification by re-reading the saved file
check = pd.read_csv(submission_path)
print("\nRe-loaded submission shape:", check.shape)
print(check.head(10))

Submission saved to /kaggle/working/submission.csv

Re-loaded submission shape: (2392, 2)
      id  emotion
0    556     fear
1   1213    happy
2    744    happy
3    443    anger
4   7673    happy
5   9373    anger
6   3506    anger
7  10699      sad
8  11929  disgust
9   4656  disgust
